# ETL de los Informes de Pedidos

Este *notebook* accede a la carpeta que posee los registros de pedidos, extrae la información de cada uno con la técnica OCR, la combina y la vuelca en una base de datos relacional que almacena como .csv.

In [1]:
# Importar librerías
import os, re, pdfplumber
import numpy as np
import pandas as pd

In [2]:
def extract_data(filename):
    """
    Extrae datos de un archivo PDF utilizando técnicas OCR y asigna cada palabra
    a su columna correspondiente basándose en las coordenadas X de posición.
    
    La función procesa todas las páginas del PDF, filtra contenido específico
    (rangos entre ciertas palabras clave) y genera tramos horarios automáticamente 
    cuando detecta formatos de hora HH:MM.

    Args:
        - filename (str): Ruta al archivo PDF a procesar (debe ser tipo .pdf)

    Returns:
        - list: Lista de tuplas con formato (texto, posición_fila, posición_columna, num_página)
              - texto (str): Palabra extraída o tramo horario generado
              - posición_fila (int): Coordenada Y redondeada de la palabra
              - posición_columna (int): Índice de columna (0-8) según coordenada X
              - num_página (int): Índice de la página (comenzando en 0)
    
    Note:
        - Filtra palabras entre "infintervalohorarioPedido" y "EUROS" (inclusive)
        - Filtra palabras entre "Leyendas" y "Mínimo" 
        - Ignora todas las instancias de la palabra "pedidos"
        - Para horas en formato HH:MM, genera tramos de 15 minutos (HH:MM - HH:MM+14)
        - Convierte comas decimales a puntos en valores numéricos
    """
    # Definir los rangos de las columnas basados en coordenadas X
    column_ranges = {
        0: (24, 75),   1: (76, 168),   2: (169, 228),
        3: (229, 288), 4: (289, 348),  5: (349, 408),
        6: (409, 468), 7: (469, 528),  8: (529, 568)
    }
    
    # Patrón precompilado para detectar formato hora
    time_pattern = re.compile(r"\d{2}:\d{2}")
    
    with pdfplumber.open(filename) as pdf:
        # Lista para almacenar todas las palabras extraídas
        selected_words = []
        
        # Iterar por todas las páginas del PDF
        for page_num, page in enumerate(pdf.pages):
            # Extraer todas las palabras de la página actual
            words = page.extract_words()  # Lista de diccionarios con datos de palabras
            
            # Variables de control para filtrado de contenido
            skip_iteration = False  # Control para saltar palabras en rangos específicos
            skip_two_iterations = False  # Control para saltar palabras después de detectar hora
            counter = 0  # Contador para el salto de iteraciones
    
            # Procesar cada palabra extraída de la página
            for word in words:
                # Manejar salto de 2 iteraciones después de procesar una hora
                if skip_two_iterations:
                    if counter == 2:
                        skip_two_iterations = False
                        counter = 0  # Resetear contador
                    else:
                        counter += 1
                        continue

                # Extraer texto y coordenada x0
                word_text = word.get("text")
                word_coord = word.get("x0")
                        
                # Ignorar palabras entre "infintervalohorarioPedido" y "EUROS" (inclusive "EUROS")
                # y entre "Leyendas" y "Mínimo" (inclusive "Mínimo")
                if word_text == "infintervalohorarioPedido" or word_text == "Leyendas":
                    skip_iteration = True
                    continue
                if word_text == "EUROS" or word_text == "Mínimo":
                    skip_iteration = False
                    continue
                    
                # Solo procesar si no estamos en modo de salto
                if not skip_iteration:
                    # Ignorar todas las instancias de "pedidos"
                    if word_text == "pedidos":
                        continue
                    
                    # Procesar palabras que no son guiones ni tienen formato de hora
                    if word_text != "-" and not time_pattern.fullmatch(word_text):
                        # Buscar en qué columna debe ir esta palabra según su coordenada X
                        for col, (min_coord, max_coord) in column_ranges.items():
                            if min_coord <= word_coord <= max_coord:
                                # Calcular posición de fila basada en coordenada Y
                                row = round(word.get("bottom"))
                                
                                # Procesar texto según su tipo (alfabético o numérico)
                                if word_text.isalpha():
                                    # Texto completamente alfabético - agregar tal como está
                                    selected_words.append((word_text, row, col, page_num))
                                else:
                                    # Texto numérico o alfanumérico - convertir comas a puntos
                                    processed_text = word_text.replace(",", ".")
                                    selected_words.append((processed_text, row, col, page_num))
                                break  # Salir del bucle una vez encontrada la columna
    
                    # Procesar formato de hora (HH:MM) y generar tramo horario
                    elif time_pattern.fullmatch(word_text):
                        # Configurar para saltar las siguientes 2 iteraciones
                        skip_two_iterations = True
                        counter = 0  # Inicializar contador
    
                        # Generar tramo horario de 15 minutos
                        hour, minute = word_text.split(":")
                        end_minute = int(minute) + 14
                        
                        # Construir tramo horario completo
                        time_slot = f"{word_text} - {hour}:{end_minute:02d}"
    
                        # Asignar tramo horario a la columna correspondiente
                        for col, (min_coord, max_coord) in column_ranges.items():
                            if min_coord <= word_coord <= max_coord:
                                row = round(word.get("bottom"))
                                selected_words.append((time_slot, row, col, page_num))
                                break  # Salir del bucle una vez encontrada la columna

    return selected_words

In [3]:
def create_dataframe(data):
    """
    Crea un DataFrame a partir de un array de tuplas que contienen datos extraídos
    de un PDF, organizándolos por filas según tramos horarios y tipos de pedido.

    La función procesa los datos para crear una estructura tabular donde:
    - Las filas representan diferentes tipos de pedido dentro de tramos horarios
    - Las columnas incluyen tramo horario, tipo de pedido y fechas como columnas de datos
    - Los valores numéricos se convierten automáticamente al tipo float

    Args:
        - data (list): Lista de tuplas con formato (texto, fila, columna, página)
                    donde cada tupla representa una palabra extraída del PDF

    Returns:
        - pd.DataFrame: DataFrame estructurado con columnas:
                     - "tramo": Tramo horario en formato "HH:MM - HH:MM"  
                     - "tipo_pedido": Tipo de pedido (texto alfabético)
                     - Columnas adicionales con fechas en formato DD/MM/YYYY
                     
    Note:
        - Filtra automáticamente filas que contienen "Total" 
        - Elimina filas desde el penúltimo "Total" hasta el final
        - Convierte valores numéricos a tipo float para cálculos posteriores
        - Agrupa datos por tramo horario y tipo de pedido
    """
    # Patrones regex precompilados
    date_pattern = re.compile(r"\d{2}/\d{2}/\d{4}")
    time_pattern = re.compile(r"\d{2}:\d{2} - \d{2}:\d{2}")
    
    # Crear columnas: comenzar con tramo e info_pedido
    columnas = ["tramo", "info_pedido"]
    
    # Agregar fechas como columnas adicionales
    i = 0
    while i < len(data) and date_pattern.fullmatch(data[i][0]):
        columnas.append(data[i][0])
        i += 1

    # Inicializar array de strings para almacenar todos los datos
    array = np.zeros([len(data), len(columnas)]).astype(str)
    
    # Contadores y variables de control
    row = -1  # Contador de filas en el array final
    vieja_info_pedido = "---"  # Variable auxiliar para detectar cambios en tipo de pedido
    
    # Procesar cada elemento de data: Se define cada fila del dataset como aquella
    # que tiene un par "hora" y "tipo de pedido" únicos. Por ejemplo, "13:00 - 13:14" y "DOMICILIO"
    # indican que toda esa fila se corresponde con datos para ese tramo horario y tipo de pedido.
    # Si el tipo de pedido cambia, por ej. "13:00 - 13:14" y "RECOGER", entonces se genera una
    # fila nueva en el dataset para esos datos. Ídem si lo que cambia es la hora.
    for x in data:
        text, _, col, _ = x  # Desempaquetar tupla
        
        # Saltar fechas (ya están en las columnas)
        if date_pattern.fullmatch(text):
            continue

        # Procesar tramos horarios
        if time_pattern.fullmatch(text):
            row += 1  # Nueva fila para nuevo tramo horario
            vieja_info_pedido = "---"  # Reset info de pedido
            array[row, col] = text # Agregar dato
            
        else:
            # Detectar info de pedido (texto alfabético) (también inicializa la primera vez)
            if text.isalpha():
                nueva_info_pedido = text
            
            # Si la info de pedido cambió, crear nueva fila
            if nueva_info_pedido != vieja_info_pedido:
                row += 1
                vieja_info_pedido = nueva_info_pedido
                array[row, col] = text
            else:
                # Mismo pedido, agregar datos a la fila actual
                array[row, col] = text
                
    # Crear DataFrame eliminando filas vacías
    array_trimmed = array[:row + 1]  # Solo filas utilizadas
    df = pd.DataFrame(array_trimmed, columns=columnas)

    # Encontrar y eliminar filas a partir del penúltimo "Total"
    total_indices = df[df["info_pedido"] == "Total"].index
    start_drop_index = total_indices[-2] + 1
    df = df.iloc[:start_drop_index].copy()  # Más eficiente que drop

    # Filtrar todas las filas que contengan "Total" 
    df = df[df["info_pedido"] != "Total"].reset_index(drop=True)

    # Renombrar columna para mayor claridad
    df.rename(columns={"info_pedido": "tipo_pedido"}, inplace=True)
    
    # Propagar tramos horarios y borrar la primera ocurrencia
    # de cada tramo (no contiene info relevante)
    rows_to_drop = []
    
    for i in range(len(df)):
        if time_pattern.fullmatch(df["tramo"][i]):
            time_slot = df["tramo"][i] # (También inicializa la primera vez)
            rows_to_drop.append(i)
        else:
            # Asignar tramo horario actual a la fila de datos
            df.iloc[i, df.columns.get_loc("tramo")] = time_slot
    
    # Eliminar filas
    df.drop(rows_to_drop, axis=0, inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Convertir columnas numéricas (fechas) a float para cálculos
    numeric_columns = df.columns.drop(["tramo", "tipo_pedido"])
    df[numeric_columns] = df[numeric_columns].astype(float)
    
    return df

In [4]:
def restructure_for_timeseries(df):
    """
    Reestructura un DataFrame de formato ancho a series temporales para análisis temporal.
    Convierte fechas de columnas a filas, combina fecha y tramo horario en timestamps
    completos, y pivota tipos de pedido como columnas separadas.
    
    Args:
        df (pd.DataFrame): DataFrame con columnas ["tipo_pedido", "tramo"] + fechas (DD/MM/YYYY)
    
    Returns:
        pd.DataFrame: DataFrame con columnas ["datetime", "fecha", "tramo"] + tipos_de_pedido,
                    donde datetime combina fecha y hora de inicio del tramo horario
    """
    # Identificar columnas de fechas
    date_columns = [col for col in df.columns if col not in ["tipo_pedido", "tramo"]]
    
    # Hacer melt para convertir las fechas en una columna
    df_melted = pd.melt(
        df, # Dataframe a transformar
        id_vars=["tipo_pedido", "tramo"], # Columnas que se mantienen como están
        value_vars=date_columns, # Columnas que se van a "derretir" en filas
        var_name="fecha", # Nombre de la nueva columna para las columnas antiguas
        value_name="cantidad" # Nombre de la nueva columna para los valores
    )
    
    # Convertir la fecha a datetime
    df_melted["fecha"] = pd.to_datetime(df_melted["fecha"], format="%d/%m/%Y")
    
    # Crear una columna datetime completa combinando fecha y tramo
    df_melted["datetime"] = df_melted.apply(lambda x: pd.to_datetime(f'{x["fecha"].date()} {x["tramo"].split("-")[0]}'), axis=1)
    
    # Pivotar los tipos de pedido a columnas
    df_final = df_melted.pivot_table(
        index=["datetime", "fecha", "tramo"], # Columnas que serán los índices
        columns="tipo_pedido", # Columna cuyos valores se convertirán en nuevas columnas. 
                               # Esto hace que "tipo_pedido" aparezca como un multiíndice 
                               # en una jerarquía superior (índice de índices). Habrá que 
                               # eliminarlo.
        values="cantidad", # Valores que llenarán las columnas
        aggfunc="sum", # Función de agregación
        fill_value=0.0 # Valor para rellenar los datos faltantes
    ).reset_index()

    # Eliminar el índice jerárquico de columnas
    df_final.columns.name = None

    return df_final

Se procede a la extracción de los registros de pedidos. Para ello se necesita acceder a una carpeta que solo contenga los informes de pedidos y en formato *.pdf*. Después, se aplicarán las funciones necesarias hasta llegar a un *dataframe* con el que se pueda trabajar.

In [5]:
# Ruta a los datos
project_root = os.getcwd()
while project_root != os.path.dirname(project_root) and not os.path.exists(os.path.join(project_root, "configs", "config.yaml")):
    project_root = os.path.dirname(project_root)
data_path = os.path.join(project_root, "data", "external", "orders_reports")

In [6]:
# Extraer todos los datos y convertirlos a dataframes
dfs = []
for filename in os.listdir(data_path):
    if filename.endswith(".pdf"):
        data = extract_data(os.path.join(data_path, filename))
        df_data = create_dataframe(data)
        dfs.append(df_data)

In [7]:
dfs[0]

,tramo,tipo_pedido,01/01/2024,02/01/2024,03/01/2024,04/01/2024,05/01/2024,06/01/2024,07/01/2024
0,13:00 - 13:14,DOMICILIO,1.0,0.0,0.0,0.0,0.0,2.0,0.0
1,13:00 - 13:14,LLEVAR,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,13:00 - 13:14,RECOGER,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3,13:15 - 13:29,DOMICILIO,3.0,1.0,0.0,1.0,0.0,1.0,2.0
4,13:15 - 13:29,LLEVAR,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...
97,22:45 - 22:59,LLEVAR,0.0,0.0,0.0,1.0,1.0,1.0,0.0
98,22:45 - 22:59,RECOGER,0.0,0.0,0.0,0.0,0.0,2.0,0.0
99,23:00 - 23:14,RECOGER,1.0,0.0,0.0,0.0,0.0,0.0,0.0
100,23:15 - 23:29,DOMICILIO,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
# Derretir y pivotar todos los dataframes
dfs_melted_pivoted = []
for df in dfs:
    dfs_melted_pivoted.append(restructure_for_timeseries(df))

In [9]:
dfs_melted_pivoted[0]

,datetime,fecha,tramo,DOMICILIO,LLEVAR,LOCAL,RECOGER
0,2024-01-01 13:00:00,2024-01-01,13:00 - 13:14,1,0,0,0
1,2024-01-01 13:15:00,2024-01-01,13:15 - 13:29,3,0,0,0
2,2024-01-01 13:30:00,2024-01-01,13:30 - 13:44,0,0,0,0
3,2024-01-01 13:45:00,2024-01-01,13:45 - 13:59,1,0,0,0
4,2024-01-01 14:00:00,2024-01-01,14:00 - 14:14,2,0,0,2
...,...,...,...,...,...,...,...
219,2024-01-07 22:15:00,2024-01-07,22:15 - 22:29,0,1,1,0
220,2024-01-07 22:30:00,2024-01-07,22:30 - 22:44,0,0,0,0
221,2024-01-07 22:45:00,2024-01-07,22:45 - 22:59,0,0,0,0
222,2024-01-07 23:00:00,2024-01-07,23:00 - 23:14,0,0,0,0


In [10]:
# Concatenar todos los dataframes
df_result = pd.concat([df for df in dfs_melted_pivoted], ignore_index=True)

In [11]:
# Ordenar el dataframe resultante por fecha
df_sorted = df_result.sort_values(by="datetime").reset_index(drop=True)

In [12]:
# Guardar df como csv
store_path = os.path.join(project_root, "data", "raw")
df_sorted.to_csv(os.path.join(store_path, "orders.csv"), index=False)